# Task 1 – NAV History Cleaning

In [7]:
import pandas as pd
import numpy as np

In [8]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

print(nav.shape)
print(nav.info())
print(nav.head())

(46000, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB
None
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [9]:
nav["date"] = pd.to_datetime(nav["date"])

nav = (
    nav
    .sort_values(["amfi_code", "date"])
    .drop_duplicates()
)

invalid_nav = nav[nav["nav"] <= 0]

print("Duplicate Rows:", nav.duplicated().sum())
print("Invalid NAV Rows:", len(invalid_nav))

Duplicate Rows: 0
Invalid NAV Rows: 0


In [10]:
cleaned_list = []

for amfi_code, group in nav.groupby("amfi_code"):

    full_dates = pd.date_range(
        start=group["date"].min(),
        end=group["date"].max(),
        freq="D"
    )

    group = (
        group
        .set_index("date")
        .reindex(full_dates)
    )

    group["amfi_code"] = amfi_code
    group["nav"] = group["nav"].ffill()

    group = (
        group
        .reset_index()
        .rename(columns={"index": "date"})
    )

    cleaned_list.append(group)

clean_nav = pd.concat(cleaned_list, ignore_index=True)

In [11]:
print("Final Shape:", clean_nav.shape)
print(clean_nav.isnull().sum())
clean_nav.to_csv(
    "../data/processed/clean_nav_history.csv",
    index=False
)
print("Saved")

Final Shape: (64320, 3)
date         0
amfi_code    0
nav          0
dtype: int64
Saved


## Summary

The NAV History dataset was cleaned and validated. The date column was converted to datetime format and datas were sorted by AMFI code and date. Nav values are verified to be greater then zero.

Missing dates, including weekends and holidays, were added for each fund, and NAV values were forward-filled using the latest available data. This increased the dataset size from 46,000 to 64,320 records.

The cleaned dataset was saved as data/processed/clean_nav_history.csv.

# Task 2 – Investor Transactions Cleaning

In [31]:
transactions = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

print("Shape:", transactions.shape)
transactions.head()

Shape: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [32]:
transactions.info()

print("\nMissing Values:")
print(transactions.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
tran

In [33]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
    .replace({"Sip": "SIP"})
)

transactions = transactions.drop_duplicates()

In [34]:
invalid_amount = transactions[
    transactions["amount_inr"] <= 0
]

print("Invalid Amount Rows:", len(invalid_amount))

print("\nTransaction Types:")
print(transactions["transaction_type"].unique())

print("\nKYC Status:")
print(transactions["kyc_status"].unique())

print("\nKYC Counts:")
print(transactions["kyc_status"].value_counts())

print("\nDuplicate Rows:",
      transactions.duplicated().sum())

Invalid Amount Rows: 0

Transaction Types:
['SIP' 'Redemption' 'Lumpsum']

KYC Status:
['Verified' 'Pending']

KYC Counts:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Duplicate Rows: 0


In [35]:
transactions.to_csv(
    "../data/processed/clean_investor_transactions.csv",
    index=False
)

print("Saved")

Saved


## Summary
The investor transactions dataset was cleaned by converting transaction dates to datetime format, standardizing transaction type values, and removing duplicate records. Data validation ensured that all transaction amounts were positive and that KYC status values were limited to the expected categories: Verified and Pending.

The cleaned dataset was then saved

# Task 3 – Scheme Performance Cleaning

In [17]:
performance = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

print("Shape:", performance.shape)

performance.head()

Shape: (40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [18]:
performance.info()

print("\nMissing Values:")
print(performance.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [28]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct"
]

for col in return_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

performance = performance.drop_duplicates()

In [29]:
invalid_expense = performance[
    (performance["expense_ratio_pct"] < 0.1)
    |
    (performance["expense_ratio_pct"] > 2.5)
]

return_anomalies = performance[
    (performance["return_1yr_pct"] > 100)
    |
    (performance["return_1yr_pct"] < -100)
]

negative_beta = performance[
    performance["beta"] < 0
]

print("Expense Ratio Anomalies:", len(invalid_expense))
print("Return Anomalies:", len(return_anomalies))
print("Negative Beta Records:", len(negative_beta))
print("Duplicate Rows:", performance.duplicated().sum())

print("\nExpense Ratio Range:")
print(
    performance["expense_ratio_pct"].min(),
    "to",
    performance["expense_ratio_pct"].max()
)

Expense Ratio Anomalies: 0
Return Anomalies: 0
Negative Beta Records: 0
Duplicate Rows: 0

Expense Ratio Range:
0.55 to 1.64


In [30]:
print("Final Shape:")
print(performance.shape)

print("\nMissing Values:")
print(performance.isnull().sum())

Final Shape:
(40, 19)

Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [37]:
performance.to_csv(
    "../data/processed/clean_scheme_performance.csv",
    index=False
)

print(
    "Saved"
)

Saved


## Summary
The scheme performance dataset was validated to ensure data accuracy and consistency. Return-related fields were confirmed as numeric, duplicate records were removed, and key performance indicators were checked for unusual values.

Expense ratios were verified to be within the expected industry range, while return metrics and beta values were reviewed for anomalies. The dataset successfully passed all validation checks and was saved